# Does Restricting Airbnb Lower Rents? Estimating Effects from NYC's Local Law 18

Writeup

New York City's housing market has faced a prolonged affordability crisis, with median rents rising sharply over the past decade and rental vacancy rates hovering near historic lows. One frequently cited reason for this pressure is the rapid growth of short-term rental platforms like Airbnb, which critics argue pull units out of the long-term rental market and drive up rents for permanent residents. This study attempts to quantify that relationship by evaluating whether the enforcement of Local Law 18, NYC's landmark short-term rental regulation, had a measurable dampening effect on rent prices across the city's zip codes.


Local Law 18 was passed by the NYC City Council in January 2022 and signed into law shortly after. Enforcement began on September 5, 2023, making New York City one of the strictest short-term rental markets in the world. The law requires all short-term rental hosts to register with the Mayor's Office of Special Enforcement, mandates that the host be physically present during any guest stay, and caps occupancy at two guests per rental. 


The data used in this analysis comes from three main sources: Zillow, the U.S. Census Bureau, and Inside Airbnb. Zillow provides the Zillow Rent Index (ZORI), which is a measure of median rent prices in various geographic areas. Inside Airbnb is a third-party data source that scrapes Airbnb and provides information on all active listings at that time. Unfortunately Inside Airbnb doesnt provide archival data without payment so we must consider some aspect of survivorship bias with our listing counts, and assume them somewhat undercounted.  I obtained ZORI data for all of New York City zip codes from January 2015 to December 2025.

### B. Findings at a Glance

** UNDER REVISION ** This study utilizes a panel dataset of Zillow Rent Index (ZRI) and Inside Airbnb listing volume across 82 NYC zip codes to evaluate the efficacy of Local Law 18. An initial Two-Way Fixed Effects (TWFE) model found no significant relationship between Airbnb intensity and rent prices ($p=0.23$). However, diagnostic testing indicated a violation of the Parallel Trends assumption ($p=0.02$ in Placebo Test), driven by differential gentrification rates in treatment zones. After correcting for these time-varying confounders using a group-specific linear time trend, the model identified a highly significant negative effect ($p<0.001$). The results suggest the policy acted as a price stabilizer, reducing rent growth by approximately 0.5% for every unit of Airbnb intensity relative to the counterfactual trend.

## II. Data

In [1]:
#Import necessary libraries
import pandas as pd
import sqlite3
import os
import matplotlib.pyplot as plt
import geopandas as gpd
import statsmodels.formula.api as smf
import numpy as np
import re
import sys
from spatial_helpers import perform_spatial_join
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML


ModuleNotFoundError: No module named 'pandas'

### A. Data Origins

In [ ]:
#Load Key Datasets 
nyc_airbnb_listings = pd.read_csv('listings.csv')
nyc_airbnb_reviews = pd.read_csv('reviews.csv')
nyc_rent_prices = pd.read_csv('Zip_zori_uc_sfrcondomfr_sm_month.csv')
total_housing_units = pd.read_csv('ACSDP5Y2023.DP04-Data.csv', skiprows=1)

The data used in this analysis comes from three main sources: Zillow, the U.S. Census Bureau, and Inside Airbnb. Zillow provides the Zillow Rent Index (ZORI), which is a measure of median rent prices in various geographic areas. Inside Airbnb is a third-party data source that scrapes Airbnb and provides information on all active listings at that time. Unfortunately Inside Airbnb doesnt provide archival data without payment so we must consider some aspect of survivorship bias with our listing counts, and assume them somewhat undercounted.   I obtained ZORI data for all of New York City zip codes from January 2015 to December 2025. The U.S. Census Bureau provides us with active housing unit counts for each zip code, which I used to normalize Airbnb listing volumes. Inside Airbnb is a third-party data source that scrapes Airbnb and provides information on all listings created. Unfortunately Inside Airbnb doesn't include data on when the listings are booked, however they do track all review activity. I used recency of reviews as a proxy for listing activity, assuming one must book a listing before leaving a review. I obtained monthly review counts for all Airbnb listings in New York City from January 2015 to December 2025.

### B. Data Processing

In [ ]:
#Cleaning Total Housing Units Data
housing_cleaned = total_housing_units.iloc[:, [1,2]].copy()
housing_cleaned.columns = ['label', 'housing_units']
housing_cleaned = housing_cleaned.rename(columns={'label': 'zip'})
housing_cleaned['zip'] = housing_cleaned['zip'].astype(str).str.replace('ZCTA5 ', '').str.strip()
housing_cleaned['housing_units'] = pd.to_numeric(housing_cleaned['housing_units'], errors='coerce')


In [ ]:
#Change nyc_rent_prices to long format and convert date column to datetime
usable_cols, dates_cols = nyc_rent_prices.columns[0:9], nyc_rent_prices.columns[9:]
nyc_rent_prices = pd.melt(nyc_rent_prices, id_vars=usable_cols, value_vars=dates_cols, var_name='date', value_name='rent_price')
nyc_rent_prices['date'] = pd.to_datetime(nyc_rent_prices['date'])

In [ ]:
#Convert review date to datetime and resample to get monthly counts
nyc_airbnb_reviews['date'] = pd.to_datetime(nyc_airbnb_reviews['date'])
monthly_counts = nyc_airbnb_reviews.resample('ME', on='date').size()

## III. Exploratory Data Analysis

### A. Temporal Patterns

#### 1. Total NYC Airbnb Review Volume Over Time

In [ ]:
#Double check time formate and plot total review volume over time

x_values = pd.to_datetime(monthly_counts.index.astype(str))
y_values = monthly_counts.values

plt.figure(figsize=(12, 6))

plt.plot(x_values, 
         y_values, 
         marker='o', 
         color='#FF5A5F', 
         linewidth=2,
         label='Monthly Reviews')

plt.xlim(pd.Timestamp('2022-01-01'), pd.Timestamp('2025-09-30'))
plt.axvline(pd.Timestamp('2023-09-05'), color='red', linestyle='--', label='Local Law 18 Enforcement')
plt.title("NYC Airbnb Review Volume Over Time", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Reviews", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout

After cleaning and merging the datasets, I started the analysis by getting a initial look at what impact Local Law 18 may have had on airbnb listings. I plotted all the monthly Airbnb reviews for all zip codes in NYC over time, with a vertical line indicating when Local Law 18 was enacted in September 2023. As you can see from the plot above, there is a clear drop in airbnb reviews after the enactment of Local Law 18. Taking into account natural cycles of travel, you can see that the number of airbnb reviews was steadily increasing before the law was enacted, and then dropped off sharply afterwards. Although it should be noted that since the law was enacted, raw review volume have reached the numbers seen 3 years prior, but not yet reaching the highs seen right before the law was enacted.

#### 2. Year-Over-Year Review Volume Changes (September Only)

In [ ]:
# Convert monthly_counts to DataFrame and filter for September only
df_monthly = pd.DataFrame({'Review Count': monthly_counts})
df_monthly.index = pd.to_datetime(df_monthly.index)

septembers = df_monthly[df_monthly.index.month == 9].copy()
septembers.index = septembers.index.year
septembers.index.name = "Year"

septembers['Change'] = septembers['Review Count'].diff()
septembers['% Change'] = septembers['Review Count'].pct_change()

styled_table = (septembers.style
    .format({
        'Review Count': '{:,.0f}',    # Comma for thousands (e.g. 15,000)
        'Change': '{:+,.0f}',         # Force +/- sign (e.g. -500)
        '% Change': '{:+.1%}'         # Percentage format (e.g. -15.5%)
    })
    .text_gradient(cmap='RdYlGn', subset=['% Change'], vmin=-0.5, vmax=0.5) # Color scale
    .set_caption("Year-Over-Year Review Changes (September Only)")
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('font-size', '12pt'), ('text-align', 'center')]}
    ])
)

# 5. Display
styled_table

This table, showing the September review counts for all zip codes in NYC, effectively shows the large year over year increase in airbnb reviews from 2009 to 2022 (excluding the 2020 pandemic). However, you can see that after the enactment of Local Law 18, the year over year growth in reviews has slowed significantly from a minimum of 30% growth from 2018-2019, to an maximum of 14.5% growth from 2024-2025. 

In [ ]:
#Create GeoDataFrame for Airbnb listing's long/lat
airbnb_gdf = gpd.GeoDataFrame(
    nyc_airbnb_listings, 
    geometry=gpd.points_from_xy(nyc_airbnb_listings.longitude, nyc_airbnb_listings.latitude),
    crs="EPSG:4326"  
)

In [ ]:
#Convert Airbnb GeoDataFrame to match NYC map CRS
nyc_map = gpd.read_file('nyc_zip_geo.geojson')
nyc_map = nyc_map.rename(columns={'label': 'zip'})

airbnb_gdf = airbnb_gdf.to_crs(nyc_map.crs)

### B. Spatial Patterns

#### 1. Baseline Airbnb Distribution

In [ ]:
# Create Plot for Baseline Airbnb Listing Volume

from spatial_helpers import process_and_merge_spatial_data

nyc_density_map = process_and_merge_spatial_data(nyc_airbnb_listings, nyc_map)

fig, ax = plt.subplots(figsize=(12, 10))
nyc_density_map.plot(
    column='count',
    cmap='OrRd',           
    linewidth=0.5,
    ax=ax,
    edgecolor='0.8',       
    legend=True,           
    legend_kwds={'label': "Number of Listings"}
)

plt.title("Baseline: Airbnb Listing Volume by Zip Code", fontsize=16)
plt.axis('off') 
plt.tight_layout()
plt.show()

Using the information provided by Inside Airbnb, I can total all listings per zip code and plot it to the a map of NYC. You can see that the spread of airbnb listings is concentrated in Manhattan and parts of Brooklyn, with some presence in Queens and the Bronx. Note that this doesn't reflect current activity, just the total number of listings that have ever been created in each zip code. However, it does give us a good idea of where airbnb activity is concentrated in NYC.

In [ ]:
#Create lookup table to merge zip codes back to reviews data

from spatial_helpers import perform_spatial_join


joined_data = perform_spatial_join(nyc_airbnb_listings, nyc_map)

zip_lookup = joined_data[['id', 'zip']].drop_duplicates()
nyc_airbnb_reviews = nyc_airbnb_reviews.merge(
    zip_lookup, 
    left_on='listing_id', 
    right_on='id', 
    how='left'
)

#### 2. Change in Airbnb Review Volume (Summer 23' vs. Summer 24') by Zip Code

In [ ]:
#Define pre- and post-law periods
nyc_airbnb_reviews['date'] = pd.to_datetime(nyc_airbnb_reviews['date'])
nyc_airbnb_reviews['month'] = nyc_airbnb_reviews['date'].dt.to_period('M')

pre_months  = pd.period_range('2023-05', '2023-08', freq='M')
post_months = pd.period_range('2024-05', '2024-08', freq='M')

pre_reviews  = nyc_airbnb_reviews[nyc_airbnb_reviews['month'].isin(pre_months)]
post_reviews = nyc_airbnb_reviews[nyc_airbnb_reviews['month'].isin(post_months)]

#Count reviews per listing in pre- and post-law periods
pre_counts = pre_reviews['listing_id'].value_counts().reset_index()
pre_counts.columns = ['id', 'pre_count']

post_counts = post_reviews['listing_id'].value_counts().reset_index()
post_counts.columns = ['id', 'post_count']

In [ ]:
#Merge counts back to main dataset for analysis
analysis_df = joined_data[['id', 'zip']].drop_duplicates().copy()
analysis_df = analysis_df.merge(pre_counts, on='id', how='left')
analysis_df = analysis_df.merge(post_counts, on='id', how='left')

#Fill NaN counts with 0 and aggregate by zip code not including listings with low pre-law review counts
analysis_df = analysis_df.fillna(0)
zip_changes = analysis_df.groupby('zip')[['pre_count', 'post_count']].sum().reset_index()
zip_changes = zip_changes[zip_changes['pre_count'] > 50].copy()

In [ ]:
#Calculate percentage and nominal changes in review counts per zip code
zip_changes['pct_change'] = (zip_changes['post_count'] - zip_changes['pre_count']) / zip_changes['pre_count']
zip_changes['nominal_change'] = zip_changes['post_count'] - zip_changes['pre_count']

In [ ]:
#Merge changes back to NYC map and plot
nyc_change_map = nyc_map.merge(zip_changes, on='zip', how='left')

nyc_change_map['pct_plot'] = nyc_change_map['pct_change'] * 100

fig, ax = plt.subplots(figsize=(14, 12))

nyc_change_map.plot(
    column='pct_plot',
    cmap='RdBu',           
    linewidth=0.5,
    ax=ax,
    edgecolor='0.6',       
    legend=True,
    legend_kwds={
        'label': "Year-Over-Year Change in Reviews (%)", 
        'shrink': 0.6,
        'format': "%.0f%%" 
    },
    missing_kwds={
        'color': 'lightgrey', 
        'label': 'Insufficient Data'
    },
    vmin=-100,  
    vmax=100    
)

plt.title("The Impact of Local Law 18: \nChange in Airbnb Review Volume (Summer '23 vs Summer '24)", fontsize=18)
plt.axis('off') 
plt.tight_layout()
plt.show()


Using the review counts, you can calculate the percent change of review volume from the summer before the law was enacted (2023) to the summer after (2024). Interestingly, you can see that the law did not have the same effect across all zip codes. The vast majority of zip codes saw a significant drop in review volumes, some saw little or no change, and roughly 10 zip codes actually saw an increase in review volume. This suggests that the law may have had different effects in different areas of the city, which is something I will explore further in the analysis.

## IV. Methodology

### A. Variable Construction

In [ ]:
#Identify active listings in the year leading up to December 2025
if isinstance(nyc_airbnb_reviews['date'].dtype, pd.PeriodDtype):
    nyc_airbnb_reviews['date'] = nyc_airbnb_reviews['date'].dt.to_timestamp(how='end')

max_date = nyc_airbnb_reviews['date'].max()
cutoff_date = max_date - pd.DateOffset(years=1)

active_reviews_2025 = nyc_airbnb_reviews[nyc_airbnb_reviews['date'] >= cutoff_date]
active_listings_2025 = active_reviews_2025[['listing_id', 'zip']].drop_duplicates(subset='listing_id')

jan_dec_2025_reviews = active_listings_2025['listing_id'].nunique()
airroi_count = 11084

# Create 2x4 comparison table
comparison_data = {
    'Review Count Proxy\n(Inside Airbnb)': [f'{jan_dec_2025_reviews:,}'],
    'Alternative Source\n(AirROI)': [f'{airroi_count:,}'],
    'Absolute\nDifference': [f'{airroi_count - jan_dec_2025_reviews:,}'],
    'Percentage\nDifference': [f'{((airroi_count - jan_dec_2025_reviews) / airroi_count) * 100:.2f}%']
}

comparison_table = pd.DataFrame(comparison_data)

# Display with styling
styled_comparison = comparison_table.style.set_properties(**{
    'text-align': 'center',
    'font-size': '11pt',
    'padding': '10px'
}).set_table_styles([ # type: ignore
    {'selector': 'th', 'props': [
        ('text-align', 'center'), 
        ('font-weight', 'bold'), 
        ('font-size', '11pt'),
        ('background-color', '#f0f0f0'),
        ('padding', '10px')
    ]},
]).hide(axis='index').set_caption('Active Listings Data Validation (Past Year to Dec 2025)')

styled_comparison


Before continuing with the analysis, I feel it would be a good idea to measure how effective our proxy for airbnb activity (review counts) is. According to AirROI, a data analytics platform tracking Airbnb activity in NYC,  "January 2025 to December 2025 [AirROI data] reveals key trends in the bustling market of 11,084 active listings". Using our method of counting an active listing as one that has at least one review in the past 12 months, you find that there are 10,789 active listings in NYC for the same time period. This means our method is under counting active listings by about 295 listings, or roughly 2.66%. While this is not a perfect match, it is close enough to give us confidence that our proxy is a reasonable measure of active Airbnb listings in NYC.

#### 1. Outcome Variable

$$\ln(Rent_{it})$$

Our outcome variable is the natural log of the Zillow Rent Index (ZRI) for zip code $i$ at time $t$. This transformation allows us to interpret coefficients as percentage changes in rent prices.

#### 2. Treatment Variable: Post-Law Dummy (Sept 2023)

$$Post_t = \begin{cases}
1 & \text{if } t \geq \text{September 2023} \
0 & \text{otherwise}
\end{cases}$$

This is a binary variable indicating whether the observation is from before or after the enactment of Local Law 18.

#### 3. Treatment Intensity: Airbnb Intensity

In [ ]:
#Calculate Airbnb intensity per zip code based on 2022 baseline
baseline_start = '2022-01-01'
baseline_end = '2022-12-31'

baseline_reviews = nyc_airbnb_reviews[
    (nyc_airbnb_reviews['date'] >= baseline_start) & 
    (nyc_airbnb_reviews['date'] <= baseline_end)
]

baseline_counts = baseline_reviews.groupby('zip')['listing_id'].nunique().reset_index()
baseline_counts.columns = ['zip', 'active_listings_2022']

baseline_counts['zip'] = baseline_counts['zip'].astype(str)
housing_cleaned['zip'] = housing_cleaned['zip'].astype(str)

intensity_df = baseline_counts.merge(
    housing_cleaned, 
    left_on='zip', 
    right_on='zip', 
    how='inner'
)

intensity_df['airbnb_intensity'] = (
    intensity_df['active_listings_2022'] / intensity_df['housing_units']
) * 1000


$$Intensity_i = \frac{\text{Active Listings}_{i,2022}}{\text{Housing Units}_i} \times 1000$$

In order to measure the intensity of Airbnb activity in each zip code, I divided the amount of active Airbnb listings by the total amount of housing units. I then multiplied by 1000 to get a more interpretable number. This allows us to capture the varying levels of Airbnb presence across different zip codes.

### B. Sample Selection

In [ ]:
#Prepare regression dataset by merging rent prices with Airbnb intensity
nyc_rent_prices['RegionName'] = nyc_rent_prices['RegionName'].astype(str)
intensity_df['zip'] = intensity_df['zip'].astype(str)

regression_df = nyc_rent_prices.merge(
    intensity_df[['zip', 'airbnb_intensity']], 
    left_on='RegionName',
    right_on='zip',
    how='inner'
)

regression_df = regression_df.drop(columns=['StateName', 'State', 'RegionType', 'RegionName', 'City', 'Metro']) # drop the duplicate


#### 1. Missing Rent Data Across Zip Codes

In order to avoid ensure robust sample size and quality, we now check to see how many zip codes have sufficient data for analysis. 

In [ ]:

# Calculate percentage of missing rent data for each zip code
missing_by_zip = regression_df.groupby('zip')['rent_price'].apply(
    lambda x: (x.isnull().sum() / len(x)) * 100
).reset_index()
missing_by_zip.columns = ['zip', 'pct_missing']

# Create histogram
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(missing_by_zip['pct_missing'], bins=20, color='steelblue', 
        edgecolor='black', alpha=0.7)

# Add threshold lines
ax.axvline(x=5, color='green', linestyle='--', linewidth=2, label='5% Threshold (Strict)')
ax.axvline(x=10, color='orange', linestyle='--', linewidth=2, label='10% Threshold (Moderate)')

ax.set_xlabel('Percent Missing Rent Data (%)', fontsize=12)
ax.set_ylabel('Number of Zip Codes', fontsize=12)
ax.set_title('Distribution of Missing Rent Data Across NYC Zip Codes (2015-2025)', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n=== SAMPLE SELECTION SUMMARY ===")
print(f"Total Zip Codes: {len(missing_by_zip)}")
print(f"\nMissingness Distribution:")
print(f"  - Perfect data (0% missing): {(missing_by_zip['pct_missing'] == 0).sum()} zips")
print(f"  - Excellent (<5% missing): {(missing_by_zip['pct_missing'] < 5).sum()} zips")
print(f"  - Good (<10% missing): {(missing_by_zip['pct_missing'] < 10).sum()} zips")
print(f"  - Problematic (>10% missing): {(missing_by_zip['pct_missing'] > 10).sum()} zips")
print(f"  - Very poor (>50% missing): {(missing_by_zip['pct_missing'] > 50).sum()} zips")

As can be seen from the histogram above, there are 61 zip codes with less than 5% missing data, and 86 with less than 10%.

#### 2. Data Cleaning and Final Sample Selection

 Going forward, my data cleaning strategy will be as follows:

1. Trim data to only include 2019-2025
2. Keep only zip codes with less than 5% missing data
3. Interpolate any remaining missing data using linear interpolation
4. Drop any remaining missing data points

In [ ]:


# Strategy 1: Trim to 2019+ (drop early years with sparse coverage)
start_date = '2019-01-01'
trimmed_df = regression_df[regression_df['date'] >= start_date].copy()

# Strategy 2: Keep only zip codes with <5% missing data
missing_vals_trimmed = trimmed_df.groupby('zip')['rent_price'].apply(lambda x: x.isnull().mean())
valid_zips = missing_vals_trimmed[missing_vals_trimmed <= 0.05].index 

# Strategy 3: Interpolate small gaps (≤2 consecutive months)
final_df = trimmed_df[trimmed_df['zip'].isin(valid_zips)].copy()
missing_before = final_df['rent_price'].isnull().sum()
final_df['rent_price'] = final_df.groupby('zip')['rent_price'].transform(
    lambda x: x.interpolate(method='linear', limit=2)
)
missing_after = final_df['rent_price'].isnull().sum()

# Strategy 4: Drop remaining NaNs
final_df = final_df.dropna(subset=['rent_price'])

# Create summary table
summary_data = {
    'Stage': [
        'Initial Sample (2015-2025)',
        'After Temporal Trim (2019+)',
        'After Zip Filter (≤5% missing)',
        'After Interpolation',
        'Final Clean Sample'
    ],
    'Zip Codes': [
        regression_df['zip'].nunique(),
        trimmed_df['zip'].nunique(),
        len(valid_zips),
        len(valid_zips),
        final_df['zip'].nunique()
    ],
    'Observations': [
        len(regression_df),
        len(trimmed_df),
        len(trimmed_df[trimmed_df['zip'].isin(valid_zips)]),
        len(trimmed_df[trimmed_df['zip'].isin(valid_zips)]) - missing_after,
        len(final_df)
    ]
}

summary_table = pd.DataFrame(summary_data)
print("=== SAMPLE CONSTRUCTION SUMMARY ===\n")
print(summary_table.to_string(index=False))
print(f"\nFinal Sample: {final_df['zip'].nunique()} zip codes × {final_df['date'].nunique()} months = {len(final_df):,} observations")

After applying our data cleaning strategy, I am left with a final sample of 82 zip codes and 6806 observations for analysis.

#### 3. Airbnb Intensity Distribution

In [ ]:
#Check the distribution of Airbnb intensity among the 82 surviving zip codes

survivors = final_df[['zip', 'airbnb_intensity']].drop_duplicates()

print("--- Survivor Stats ---")
print(survivors['airbnb_intensity'].describe())

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(survivors['airbnb_intensity'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Airbnb Intensity in the 82 Survivor Zips')
plt.xlabel('Active Listings per 1,000 Units')
plt.ylabel('Count of Zip Codes')
plt.axvline(survivors['airbnb_intensity'].median(), color='red', linestyle='dashed', label='Median')
plt.legend()
plt.show()

print("\nLowest Intensity Survivors (Control Group):")
print(survivors.sort_values('airbnb_intensity').head(5))

print("\nHighest Intensity Survivors (Treatment Group):")
print(survivors.sort_values('airbnb_intensity').tail(5))

### C. Econometric Framework

$$\ln(Rent_{it}) = \beta_0 + \beta_1(Intensity_i \times Post_t) + \alpha_i + \gamma_t + \epsilon_{it}$$


$\alpha_i$ = zip code fixed effects (controls for time-invariant characteristics)
$\gamma_t$ = month fixed effects (controls for city-wide trends)
$\epsilon_{it}$ = error term, clustered by zip code
$\beta_1$ = difference-in-differences estimate (our parameter of interest)

Interpretation of $\beta_1$: For each additional Airbnb listing per 1,000 housing units, the percentage change in rent after the law relative to before.

In [ ]:

#Log transform rent prices, create time dummy, and run TWFE regression
final_df['log_rent'] = np.log(final_df['rent_price'])

final_df['post_law'] = (final_df['date'] >= '2023-09-01').astype(int)

model_TWFE = smf.ols(
    formula = "log_rent ~ airbnb_intensity:post_law + C(zip) + C(date)", 
    data = final_df
).fit()


## V. Initial Results

### A. Baseline TWFE Model Results

In [ ]:
#Create a nice and pretty regression table

stargazer = Stargazer([model_TWFE])

stargazer.custom_columns(['Two-Way Fixed Effects'], [1])  
stargazer.show_model_numbers(False)                       
stargazer.significant_digits(4)                           
stargazer.show_degrees_of_freedom(False)                  
stargazer.covariate_order(['airbnb_intensity:post_law'])  
stargazer.rename_covariates({'airbnb_intensity:post_law': 'Airbnb Intensity x Post-Law'})
stargazer.add_line('P-Value', [f"{model_TWFE.pvalues['airbnb_intensity:post_law']:.4f}"])
stargazer.add_line('Zip Code Fixed Effects', ['Yes'])
stargazer.add_line('Month Fixed Effects', ['Yes'])

HTML(stargazer.render_html())

As can be seen from the table above, our initial TWFE model shows a one-unit increase in Airbnb intensity is associated with a 0.04% decrease in rents. A negligible effect, and not statistically significant ($p=0.2$). Our model also seems to explain much of the variation in rent prices, with an adjusted $R^2$ of .99. Assuming that the zip code fixed effects are doing most of the work. To account for serial correlation between observations, I will use clustered standard errors around zip codes in our next regression below.

### B. Robustness Check: Clustered Standard Errors

In [ ]:
model_TWFE_cluster = smf.ols(
    formula = "log_rent ~ airbnb_intensity:post_law + C(zip) + C(date)", 
    data = final_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': final_df['zip']}
)

# Create dual model comparison
stargazer = Stargazer([model_TWFE, model_TWFE_cluster])

# Customization
stargazer.custom_columns(['Standard SE', 'Clustered SE'], [1, 1])
stargazer.show_model_numbers(False)
stargazer.significant_digits(4)
stargazer.show_degrees_of_freedom(False)
stargazer.covariate_order(['airbnb_intensity:post_law'])
stargazer.rename_covariates({'airbnb_intensity:post_law': 'Airbnb Intensity × Post-Law'})

# Add p-values explicitly for both models
stargazer.add_line('P-Value', [
    f"{model_TWFE.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{model_TWFE_cluster.pvalues['airbnb_intensity:post_law']:.4f}"
])

# Add fixed effects
stargazer.add_line('Zip Code Fixed Effects', ['Yes', 'Yes'])
stargazer.add_line('Month Fixed Effects', ['Yes', 'Yes'])
stargazer.add_line('Clustered by Zip', ['No', 'Yes'])

HTML(stargazer.render_html())

After clustering standard errors to account for serial correlation it seems that our model still has the same $R^2$. However, our standard error has increased by over 400%. This suggests that our original model had confidence intervals that were too narrow, and I should use clustered standard errors moving forward. 

### C. Subgroup Analysis: Luxury Market

In [ ]:
#Define and test 'Luxury Sub-Market' based on 2022 median rents
avg_rents_2022 = final_df[final_df['date'].dt.year == 2022].groupby('zip')['rent_price'].median()
expensive_threshold = avg_rents_2022.quantile(0.75)
luxury_zips = avg_rents_2022[avg_rents_2022 >= expensive_threshold].index
print(f"Testing the 'Luxury Sub-Market' (N={len(luxury_zips)} zips)...")

In [ ]:
# Subset final_df to luxury zips and rerun regression
luxury_df = final_df[final_df['zip'].isin(luxury_zips)].copy()

luxury_model = smf.ols(
    formula = "log_rent ~ airbnb_intensity:post_law + C(zip) + C(date)", 
    data = luxury_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': luxury_df['zip']}
)

In [ ]:
# Create three-model comparison
stargazer = Stargazer([model_TWFE, model_TWFE_cluster, luxury_model])

# Customization
stargazer.custom_columns(['Standard SE', 'Clustered SE', 'Luxury Submarket'], [1, 1, 1])
stargazer.show_model_numbers(False)
stargazer.significant_digits(4)
stargazer.show_degrees_of_freedom(False)
stargazer.show_f_statistic = False
stargazer.covariate_order(['airbnb_intensity:post_law'])
stargazer.rename_covariates({'airbnb_intensity:post_law': 'Airbnb Intensity × Post-Law'})

# Add p-values explicitly for all three models
stargazer.add_line('P-Value', [
    f"{model_TWFE.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{model_TWFE_cluster.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{luxury_model.pvalues['airbnb_intensity:post_law']:.4f}"
])

# Add fixed effects and notes
stargazer.add_line('Zip Code Fixed Effects', ['Yes', 'Yes', 'Yes'])
stargazer.add_line('Month Fixed Effects', ['Yes', 'Yes', 'Yes'])
stargazer.add_line('Clustered by Zip', ['No', 'Yes', 'Yes'])
stargazer.add_line('Sample', ['All Zips', 'All Zips', 'Top 25% Rent'])

HTML(stargazer.render_html())

Considering most Airbnbs are located within higher income areas, I should test to see if the effect of the ban can be seen in these neighborhoods. To do this I took an average rent price of each zip code in 2022, and retested the previous model with the top 25% of zip codes. The results show a coefficient thats stronger than than our other models, but still with a p-value thats too large to find significant. 

## VI. Diagnostic Testing

### A. Event Study Analysis

In [ ]:
# 1. Setup: Calculate 'Month Distance' from Sep 2023
final_df['date'] = pd.to_datetime(final_df['date'])
final_df['months_diff'] = (
    (final_df['date'].dt.year - 2023) * 12 + 
    (final_df['date'].dt.month - 9)
)

# 2. Create Interaction Terms (Intensity * Month Relative to Law)
interaction_cols = []
# Range: 2 years pre-law (-24) to 1 year post-law (+12)
for k in range(-24, 13):
    # Skip k=-1 (August 2023) to serve as the reference period (coefficient = 0)
    if k == -1:
        continue
        
    suffix = "neg" if k < 0 else "pos"
    col_name = f"int_{suffix}_{abs(k):02d}"
    
    # Interaction: Airbnb Intensity * Indicator for this specific relative month
    final_df[col_name] = final_df['airbnb_intensity'] * (final_df['months_diff'] == k).astype(int)
    interaction_cols.append(col_name)

In [ ]:
# 3. Run Regression with Fixed Effects
# Formula: log_rent ~ (Intensity * Month_Dummies) + Zip_FE + Month_FE
interaction_formula = " + ".join(interaction_cols)
formula = f"log_rent ~ {interaction_formula} + C(zip) + C(date)"

# Setup Model, using clustered standard errors by Zip to account for serial correlation in rents.
model = smf.ols(formula, data=final_df).fit(
    cov_type='cluster', 
    cov_kwds={'groups': final_df['zip']}
)

# 4. Extract Results for Plotting
results = []
for k in range(-24, 13):
    # Manually insert the reference point
    if k == -1:
        results.append({'month': -1, 'coef': 0, 'err': 0, 'ci_lower': 0, 'ci_upper': 0})
        continue
        
    suffix = "neg" if k < 0 else "pos"
    col_name = f"int_{suffix}_{abs(k):02d}"
    
    if col_name in model.params:
        coef = model.params[col_name]
        err = model.bse[col_name] # Robust standard error
        results.append({
            'month': k,
            'coef': coef,
            'err': err,
            'ci_lower': coef - 1.96*err, # 95% Confidence Interval
            'ci_upper': coef + 1.96*err
        })

results_df = pd.DataFrame(results).sort_values('month')

In [ ]:
def plot_event_study(df):
    plt.figure(figsize=(10, 6))
    
    # Plot the coefficients
    plt.errorbar(
        df['month'], 
        df['coef'], 
        yerr=df['err']*1.96, # 95% CI
        fmt='o', 
        color='b', 
        ecolor='gray', 
        capsize=3,
        label='Coefficient (95% CI)'
    )
    
    # Add reference line at y=0 and x=-1
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    plt.axvline(x=-1, color='g', linestyle=':', alpha=0.5, label='Treatment Start')
    
    plt.title('Event Study: Coefficients over Time')
    plt.xlabel('Months relative to event')
    plt.ylabel('% Difference in Log Rent Between Treatment and Control zips')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
plot_event_study(results_df)

To see if the null result I found testing our past models was accurate, I decided to verify our pre-trend lines were parallel. In order for our difference-in-difference model to detect any effect from the law, our treatment group (high-rent zips) and control group (low-rent zips) must be moving in parallel before the law passed. I mapped out the $Post=Law$ variable into month-by-month coefficients so you can see the difference in rent prices between groups leading up to and after the law passes.

The event study reveals a stark violation of this assumption as you can clearly the difference between neighborhoods steadily increases up until 14 months before the law was passed, with the difference naturally cooling off from that point until then. To verify the parallel lines assumption was violated, i'll run a placebo test below.

### B. Placebo Test

In [ ]:
#Create placebo test by defining fake law implementation date in 2019
placebo_df = final_df[final_df['date'] < '2022-01-01'].copy()
placebo_df['fake_law'] = (placebo_df['date'] >= '2019-09-01').astype(int)


#Run placebo regression

placebo_model = smf.ols(
    formula = "log_rent ~ airbnb_intensity:fake_law + C(zip) + C(date)", 
    data = placebo_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': placebo_df['zip']}
)

print(f"Placebo Test (2019) P-Value: {placebo_model.pvalues['airbnb_intensity:fake_law']:.4f}")


In [ ]:
# Subset final_df to luxury zips and rerun regression
luxury_df = final_df[final_df['zip'].isin(luxury_zips)].copy()

luxury_model = smf.ols(
    formula = "log_rent ~ airbnb_intensity:post_law + C(zip) + C(date)", 
    data = luxury_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': luxury_df['zip']}
)

# Create four-model comparison
stargazer = Stargazer([model_TWFE, model_TWFE_cluster, luxury_model, placebo_model])

# Customization
stargazer.custom_columns(['Standard SE', 'Clustered SE', 'Luxury Submarket', 'Placebo (2019)'], [1, 1, 1, 1])
stargazer.show_model_numbers(False)
stargazer.significant_digits(4)
stargazer.show_degrees_of_freedom(False)
stargazer.show_f_statistic = False
stargazer.covariate_order(['airbnb_intensity:post_law', 'airbnb_intensity:fake_law'])
stargazer.rename_covariates({
    'airbnb_intensity:post_law': 'Airbnb Intensity × Post-Law',
    'airbnb_intensity:fake_law': 'Airbnb Intensity × Fake Law (2019)'
})

# Add p-values explicitly for all four models
stargazer.add_line('P-Value', [
    f"{model_TWFE.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{model_TWFE_cluster.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{luxury_model.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{placebo_model.pvalues['airbnb_intensity:fake_law']:.4f}"
])

# Add fixed effects and notes
stargazer.add_line('Zip Code Fixed Effects', ['Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Month Fixed Effects', ['Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Clustered by Zip', ['No', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Sample', ['All Zips', 'All Zips', 'Top 25% Rent', 'Pre-2022 Only'])

HTML(stargazer.render_html())

To verify the parallel lines assumption was violated, I ran a placebo test on the model. Using only data from 2022 and before, I reran the model but set the law implementation date in 2019. Since there was obviously no airbnb law passed in 2019, our model should not be able to detect any effect this non existent law had. However, as you can see from the regression table above, our placebo test was $\text{97}\%$ confident our fake law had a $\text{.33}\%$ change in rent prices. In conclusion, our model shows a fundamental error in its inability to distinguish the laws effect from natural cycles. To account for this I must create a model that explicitly controls for divergent neighborhood trends. 

## VII. Model Refinement

### A. Group-Specific Time Trends

In [ ]:
#Change, create a time trend variable

final_df['time_trend'] = (final_df['date'].dt.year - final_df['date'].dt.year.min()) * 12 + \
                         (final_df['date'].dt.month - final_df['date'].dt.month.min())

$$\ln(Rent_{it}) = \beta_0 + \beta_1(Intensity_i \times Post_t) + \beta_2(Intensity_i \times TimeTrend_{it}) + \alpha_i + \gamma_t + \epsilon_{it}$$

The addition of $\beta_2(Intensity_i \times TimeTrend_{it})$ allows zip codes with higher Airbnb intensity to have different baseline rent growth trajectories, addressing violations of the parallel trends assumption.

In [ ]:
#Correct for potential time-varying confounders with interaction term
corrected_model = smf.ols(
    formula = "log_rent ~ airbnb_intensity:post_law + airbnb_intensity:time_trend + C(zip) + C(date)", 
    data = final_df
).fit(
    cov_type='cluster',
    cov_kwds={'groups': final_df['zip']}
)

### B. Corrected Model Results

In [ ]:
# Create five-model comparison
stargazer = Stargazer([model_TWFE, model_TWFE_cluster, luxury_model, placebo_model, corrected_model])

# Customization
stargazer.custom_columns(['Standard SE', 'Clustered SE', 'Luxury Submarket', 'Placebo (2019)', 'Trend-Corrected'], [1, 1, 1, 1, 1])
stargazer.show_model_numbers(False)
stargazer.significant_digits(4)
stargazer.show_degrees_of_freedom(False)
stargazer.show_f_statistic = False
stargazer.covariate_order(['airbnb_intensity:post_law', 'airbnb_intensity:fake_law', 'airbnb_intensity:time_trend'])
stargazer.rename_covariates({
    'airbnb_intensity:post_law': 'Airbnb Intensity × Post-Law',
    'airbnb_intensity:fake_law': 'Airbnb Intensity × Fake Law (2019)',
    'airbnb_intensity:time_trend': 'Airbnb Intensity × Time Trend'
})

# Add p-values explicitly for all five models
stargazer.add_line('P-Value (Treatment)', [
    f"{model_TWFE.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{model_TWFE_cluster.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{luxury_model.pvalues['airbnb_intensity:post_law']:.4f}",
    f"{placebo_model.pvalues['airbnb_intensity:fake_law']:.4f}",
    f"{corrected_model.pvalues['airbnb_intensity:post_law']:.4f}"
])

# Add fixed effects and notes
stargazer.add_line('Zip Code Fixed Effects', ['Yes', 'Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Month Fixed Effects', ['Yes', 'Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Clustered by Zip', ['No', 'Yes', 'Yes', 'Yes', 'Yes'])
stargazer.add_line('Sample', ['All Zips', 'All Zips', 'Top 25% Rent', 'Pre-2022 Only', 'All Zips'])

HTML(stargazer.render_html())

After including $\beta_2(Intensity_i \times TimeTrend_{it})$ you can see that our $\text{Airbnb Intensity} \times \text{Post-Law}$ coefficient, or estimated impact of the airbnb law on rent prices, is 12 times higher than previously estimated. And unlike our previous models, our finding is highly significant with a $p < 0.01$. Our new variable $\beta_2(Intensity_i \times TimeTrend_{it})$ has a coefficient of 0.0001, showing that Air-bnb heavy neighborhoods were naturally getting more expensive over time.

Our newly created model that includes the time trend interaction estimates: for every additional listing per 1,000 units, rent growth slowed by ~0.5% post-law. To use an example, zip-code 10018 had an Airbnb intensity metric of ~10. Meaning that roughly every 10 housing units out of 1,000 are airbnb listings. This would mean our model indicates that Local Law 18 enforcement has a cumulative effect of -4.9% within this zip-code. Or in other words, our model estimates **rent prices would be ~5% higher in this zip code if this law hadn't passed.**

## VIII. Robustness Checks

In order to further verify our findings I will run two robustness checks

1. Verifying the time-trend fix actually eliminated the bias
2. Confirming the result isn't driven by a few highly concentrated zips.


### A. Re-Placebo Test

If our trend correction worked, the model should now fail to detect the fake 2019 law that tripped it up before. A pass here validates the correction.

In [ ]:
# Run trend-corrected model on pre-2022 data with fake 2019 law
placebo_df = final_df[final_df['date'] < '2022-01-01'].copy()
placebo_df['fake_law'] = (placebo_df['date'] >= '2019-09-01').astype(int)
placebo_df['time_trend'] = (placebo_df['date'].dt.year - placebo_df['date'].dt.year.min()) * 12 + \
                           (placebo_df['date'].dt.month - placebo_df['date'].dt.month.min())

replacebo_model = smf.ols(
    formula = "log_rent ~ airbnb_intensity:fake_law + airbnb_intensity:time_trend + C(zip) + C(date)", 
    data = placebo_df
).fit(cov_type='cluster', cov_kwds={'groups': placebo_df['zip']})

print(f"Re-Placebo P-Value: {replacebo_model.pvalues['airbnb_intensity:fake_law']:.4f}")

After running our placebo test again I found that our model no longer hallucinates an effect, confirming the time-trend interaction successfully absorbed the divergent gentrification trends. In other words, our model seems to have now passed the placebo test. 

### B. Outlier Sensitivity Check

Due to New York's concentration of Airbnb activity, its wise to see what changes occur when excluding the zip codes with the highest airbnb intensity. I will drop each of the top three zip codes individually to see if the coefficient holds.

In [ ]:
# Drop top 3 highest-intensity zip codes one at a time and recheck the coefficient
top_zips = final_df.groupby('zip')['airbnb_intensity'].max().sort_values(ascending=False).head(3).index.tolist()
base_coef = corrected_model.params['airbnb_intensity:post_law']

print(f"Baseline — Coef: {base_coef:.5f}  Zips dropped: None")
for zip_code in top_zips:
    subset_df = final_df[final_df['zip'] != zip_code].copy()
    subset_model = smf.ols(
        formula = "log_rent ~ airbnb_intensity:post_law + airbnb_intensity:time_trend + C(zip) + C(date)",
        data = subset_df
    ).fit(cov_type='cluster', cov_kwds={'groups': subset_df['zip']})

    coef = subset_model.params['airbnb_intensity:post_law']
    pval = subset_model.pvalues['airbnb_intensity:post_law']
    change = ((coef - base_coef) / base_coef) * 100
    print(f"Drop {zip_code}   — Coef: {coef:.5f}  p={pval:.3f}  Change: {change:+.1f}%")

As you can see from the results, dropping each of the top 3 zip codes resulted in at most a 2% difference in coefficients, while still remaining highly significant. This confirms our results aren't likely an artifact from a single outlier zipcode. 

## IX. Final Results & Conclusion

In [ ]:
# Identify highest-intensity zip code for case study
top_zip_row = final_df.sort_values('airbnb_intensity', ascending=False).iloc[0]
target_zip = top_zip_row['zip']
target_intensity = top_zip_row['airbnb_intensity']

### A. Case Study: Williamsburg (Zip 11211)

To put a dollar figure on the effect, I plotted the corrected model's fitted rent trajectory against the counterfactual — what the model predicts rents would have been in zip 11211 had the law never passed. Williamsburg is the highest-intensity zip in our sample (12.8 active listings per 1,000 units), making the estimated effect largest and most visible there.

In [ ]:
plot_df = final_df[final_df['zip'] == target_zip].sort_values('date').copy()

# Scenario A: with law (actual fitted values from corrected model)
plot_df['log_pred_actual'] = corrected_model.predict(plot_df)

# Scenario B: counterfactual — set post_law=0 to remove the treatment effect
counterfactual_df = plot_df.copy()
counterfactual_df['post_law'] = 0
plot_df['log_pred_counter'] = corrected_model.predict(counterfactual_df)

# Convert back to dollar rent
plot_df['rent_fitted']        = np.exp(plot_df['log_pred_actual'])
plot_df['rent_counterfactual'] = np.exp(plot_df['log_pred_counter'])

In [ ]:

# Create Plot
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(plot_df['date'], plot_df['rent_counterfactual'],
        linestyle='--', color='gray', linewidth=2, label='Projected Rent (Without Law)')
ax.plot(plot_df['date'], plot_df['rent_fitted'],
        color='darkred', linewidth=3, label='Estimated Rent (With Law)')

# Fill difference with greeen.
mask = plot_df['date'] >= '2023-09-01'
ax.fill_between(plot_df.loc[mask, 'date'],
                plot_df.loc[mask, 'rent_fitted'],
                plot_df.loc[mask, 'rent_counterfactual'],
                color='green', alpha=0.2, label='Estimated Tenant Savings')

# Law Enacted line
ax.axvline(pd.to_datetime('2023-09-01'), color='black', linestyle=':', linewidth=2)
ax.text(pd.to_datetime('2023-09-01'), plot_df['rent_fitted'].min(),
        '  Law Enacted (Sept 2023)', verticalalignment='bottom', fontsize=10)

# Calculate Savings
last_val = plot_df.iloc[-1]
savings = last_val['rent_counterfactual'] - last_val['rent_fitted']

ax.set_title(f"Local Law 18 Impact: Zip {target_zip} (Williamsburg)\nEstimated Monthly Savings by Dec 2025: ${savings:.0f}", fontsize=15)
ax.set_ylabel("Rent Price ($)", fontsize=12)
ax.set_xlabel("Date", fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('scenario_plot.png', dpi=150)
plt.show()

The green shaded region shows the gap between the estimated rent trajectory under Local Law 18 and what the model projects rents would have been without it. The gap widens gradually after September 2023 as the policy effect compounds over time. I projected what rents would look like without Local Law 18, and therefore the estimated savings, by applying our model's coefficient to zip 11211's high intensity score of 12.8 listings per 1000 units. 

### B. City-Wide Estimated Rent Savings

The Williamsburg case study shows the largest possible effect in our sample. To see how that effect varies across the city, I applied the corrected model coefficient to each zip code's Airbnb intensity score and see the implied monthly savings relative to the counterfactual. Zip codes with low Airbnb intensity will show near-zero savings, while the high-intensity clusters in Manhattan and North Brooklyn should see the largest effects. Closely mirroring the heat map seen in III B. 2.

In [ ]:
beta1 = corrected_model.params['airbnb_intensity:post_law']

# Get latest actual rent and intensity for each zip
latest_rents = final_df.groupby('zip')['rent_price'].last().reset_index()
latest_rents.columns = ['zip', 'latest_rent']
zip_intensity = final_df[['zip', 'airbnb_intensity']].drop_duplicates()

savings_df = latest_rents.merge(zip_intensity, on='zip')

# Counterfactual rent = actual * exp(-beta1 * intensity)
# Since beta1 < 0, -beta1 > 0, so counterfactual > actual (law held rents down)
savings_df['monthly_savings'] = savings_df['latest_rent'] * (np.exp(-beta1 * savings_df['airbnb_intensity']) - 1)

# Merge to map
savings_map = nyc_map.merge(savings_df, on='zip', how='left')

fig, ax = plt.subplots(figsize=(12, 10))
savings_map.plot(
    column='monthly_savings',
    cmap='YlGn',
    linewidth=0.5,
    ax=ax,
    edgecolor='0.8',
    legend=True,
    legend_kwds={'label': 'Estimated Monthly Rent Savings ($)', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey', 'label': 'Outside Sample'}
)
plt.title("Estimated Monthly Rent Savings from Local Law 18\nby NYC Zip Code (vs. Counterfactual Trend)", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Median estimated monthly savings: ${savings_df['monthly_savings'].median():.2f}")
print(f"Max estimated monthly savings:    ${savings_df['monthly_savings'].max():.2f}  (zip {savings_df.loc[savings_df['monthly_savings'].idxmax(), 'zip']})")

The map confirms that the estimated savings are geographically concentrated exactly where you would expect: the highest-density Airbnb corridors in lower Manhattan and North Brooklyn. Zip codes where there wasn't much Airbnb activity to begin with saw little to no change at all. 

**Limitations.** Several caveats apply to these estimates. First, this model assumes a linear relationship between intensity and rent. This may be a simplification that could understate effects in hyper-concentrated zip codes. Second, I estimated Airbnb activity proxy (review counts) undercounts active listings by roughly 2.7%, introducing a small downward bias on the intensity variable. Third, the analysis covers only the 82 zip codes with sufficient Zillow rent data. Some of the zip codes excluded due to lack of data tend to be lower-income, outer-borough areas where the law's effects may differ. Finally, our data runs through December 2025, capturing only ~27 months of post-enforcement data. To see if this detected stabilizing effect persists or fades as the market adjusts, a longer panel will be needed. 

### C. Conclusion

This study set out to estimate whether NYC's Local Law 18, an effective airbnb ban that had a sizeable impact on NYC airbnb rental supply, had a measurable effect on rent prices across the city's zip codes. An initial Two-Way Fixed Effects model found no significant relationship ($p=0.23$), but with further testing a violation of the parallel trends assumption revealed itself. Zip codes with higher concentrations of Airbnb's were already becoming more expensive at a faster rate than other zip codes, affecting our model's ability to capture the effect of the law change. To fix this issue I introduced a group-specific linear time trend to absorb divergent trajectories which led to my model to identify a highly significant effect ($p<0.001$). This was confirmed with a placebo test ($p=0.077$), as well as an outlier sensitivity test showing a max coefficient shift of $2.4\%$. My end result suggests Local Law 18 acted as a brake for rent growth, reducing rents by approximately **0.5% per unit of Airbnb intensity** relative to the counterfactual trend. In other words, the model suggests that Local Law 18 is responsible for a median monthly rent decrease of $\sim\$50$ across all 82 zip codes tested, with some estimated zip code monthly savings as high as $\sim\$300$.